# WAXAL ASR — full pipeline (Kaggle GPU)

MMS-1b + Whisper-turbo + KenLM + LID ensemble for Luganda / Lingala / Shona.
Set **Accelerator = GPU (P100 or T4 x2)** and **Internet = ON**.
Attach the Zindi challenge files as a dataset at `/kaggle/input/waxal-challenge`.
See `STRATEGY.md` for the reasoning and `README.md` for details.

## 0. Get the code + install deps

In [ ]:
# Option A: clone from your repo. Option B: attach this project as a dataset.
import os, sys, subprocess
REPO = "/kaggle/working/waxal-asr"
if not os.path.exists(REPO):
    # If you pushed it to GitHub, clone it; otherwise copy from an attached dataset.
    # subprocess.run(["git","clone","https://github.com/<you>/waxal-asr", REPO])
    subprocess.run(["cp","-r","/kaggle/input/waxal-asr", REPO])
sys.path.insert(0, REPO); os.chdir(REPO)
print(os.listdir(REPO))

In [ ]:
%%capture
!pip install -q "transformers>=4.44" datasets accelerate peft jiwer librosa soundfile pyctcdecode
!pip install -q https://github.com/kpu/kenlm/archive/master.zip

## 1. Cache cleaned data (ONCE) — then save output as dataset `waxal-clean`

In [ ]:
!python scripts/prepare_data.py

## 2. Build KenLM 5-gram per language

In [ ]:
%%capture
!apt-get -qq install -y build-essential cmake libboost-all-dev libeigen3-dev zlib1g-dev
!git clone -q https://github.com/kpu/kenlm.git /kaggle/working/kenlm
!cd /kaggle/working/kenlm && mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release >/dev/null && make -j4 >/dev/null

In [ ]:
!python scripts/build_lm.py --kenlm_bin /kaggle/working/kenlm/build/bin

## 3. Train MMS-1b adapters (one per language)

In [ ]:
!python scripts/train_mms.py --lang lin

In [ ]:
!python scripts/train_mms.py --lang sna

In [ ]:
!python scripts/train_mms.py --lang lug

## 4. Train joint Whisper-turbo (add --lora if you OOM)

In [ ]:
!python scripts/train_whisper.py

## 5. Language ID (validate on Phase 1, run on test)

In [ ]:
!python scripts/lid.py --split validation

In [ ]:
!python scripts/lid.py --split test

## 6. Validate ensemble locally BEFORE submitting

In [ ]:
!python scripts/infer.py --model mms     --split validation
!python scripts/infer.py --model whisper --split validation
!python scripts/ensemble.py --split validation

## 7. Predict test + build submission

In [ ]:
!python scripts/infer.py --model mms     --split test
!python scripts/infer.py --model whisper --split test
!python scripts/ensemble.py --split test --route artifacts/route.json

In [ ]:
import pandas as pd
sub = pd.read_csv("artifacts/submission.csv"); print(sub.shape); sub.head()